# KG1 V80 MEGA V3 - TODOS OS FIXES CONSOLIDADOS

## Execução ultra-simplificada
**APENAS 1 célula. Execute e espera ~3h.**

O que faz automaticamente:
1. Uninstall torchcodec/torchao/torchdata (Colab pre-installed, conflita com torch 2.5)
2. Install torch 2.5.1+cu124 via wheels diretos (bypass --index-url issues)
3. Install mamba-ssm 2.2.4 + causal-conv1d 1.5.0.post8 (torch 2.5 ABI)
4. Install transformers/peft/trl/accelerate/datasets/bitsandbytes
5. Install Unsloth com --no-deps (não upgrade torch)
6. Verify via child process (torch 2.5 clean import)
7. Download colab_mega_v80.py do GitHub
8. Executa training em subprocess:
   - Dataset dgxchen v7 EXACT (problem_ids_matched.csv, 7830 rows)
   - Model Nemotron-3-Nano-30B-A3B-BF16 (cached)
   - LoRA r=32 alpha=32 dropout=0, 8 targets SEM lm_head
   - max_length=**3072** (p99 safe, otimizado H100 80GB)
   - Train 1 epoch 245 steps
   - Save adapter + Build submission.zip + HF upload + Kaggle submit

**Tempo total**: ~3-4h (com cache) ou ~4-5h (cold start)

## Todos os 13 fixes aplicados

### 7 divergências dgxchen v7 revertidas:
1. Dataset `problem_ids_matched.csv` (não less_cot.csv)
2. attn_implementation='eager' (não sdpa)
3. LoRA 8 targets SEM lm_head
4. num_train_epochs=1 (não 2)
5. max_grad_norm=1e9 (efetivamente disabled)
6. gradient_checkpointing=True + use_reentrant=False
7. formatting_func no trainer com conversation wrap

### 4 fixes execução (descobertos hoje 22/04):
8. mamba-ssm + causal-conv1d install explicit (NemotronH requer)
9. torch 2.5.1 pin via WHEELS DIRETOS (bypass --index-url bug Colab cp312)
10. dataloader_num_workers=0 (prev pickle CudaDeviceProperties error)
11. Unsloth --no-deps + uninstall torchcodec (prev ABI mismatch torch 2.11)

### 2 fixes performance (descobertos no primeiro run):
12. MAX_SEQ_LEN=**3072** (não 4096) — evita gradient offloading, 4x speedup
13. PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True (anti-fragmentation)

## Credenciais (Colab Secrets 🔒)
Adicione os 3 na barra lateral (ícone cadeado):
- `HF_KEY` = (seu HF token DEV atual)
- `KAGGLE_USERNAME` = felipe1983
- `KAGGLE_KEY` = (do kaggle.json)

## Hardware
- **Recomendado**: Colab Pro+ **H100 80GB HBM3**
- Também funciona: A100 80GB
- **Não funciona**: T4/L4/A10 (VRAM insuficiente para Nemotron-30B + MoE LoRA)

## Expected outcome
Score Kaggle: **0.84-0.85** (replica dgxchen v7 EXACT com 0.85 LB verificado 22/04/2026)

## Credenciais (Colab Secrets - icone cadeado no painel esquerdo)
Adicione estes 3 secrets no Colab (Runtime > Secrets):
- `HF_KEY` = (seu HF token DEV)
- `KAGGLE_USERNAME` = felipe1983
- `KAGGLE_KEY` = (do kaggle.json)

## Hardware
- **Recomendado**: Colab Pro+ H100 80GB HBM3
- Também funciona: A100 80GB
- **Não funciona**: T4, L4, A10 (VRAM insuficiente para 30B BF16)


In [1]:
# V80 MEGA CELL - install + download + train + submit (tudo em 1 cell)
# ATENCAO: espera ~3-4h para terminar. Output streams aqui live.
import subprocess, sys, os, json, urllib.request
from pathlib import Path

print('=' * 70)
print('V80 MEGA CELL - dgxchen v7 EXACT, 1 cell, no restart')
print('=' * 70)

# ============ 1. Load Colab secrets ============
try:
    from google.colab import userdata
    hf_key = userdata.get('HF_KEY')
    kaggle_user = userdata.get('KAGGLE_USERNAME')
    kaggle_key = userdata.get('KAGGLE_KEY')
except Exception:
    hf_key = os.environ.get('HF_KEY') or os.environ.get('HF_TOKEN')
    kaggle_user = os.environ.get('KAGGLE_USERNAME')
    kaggle_key = os.environ.get('KAGGLE_KEY')

assert hf_key, 'HF_KEY missing - add in Colab Secrets (cadeado esquerda)'
assert kaggle_user and kaggle_key, 'KAGGLE_USERNAME / KAGGLE_KEY missing'

os.environ['HF_TOKEN'] = hf_key
os.environ['HF_KEY'] = hf_key
os.environ['KAGGLE_USERNAME'] = kaggle_user
os.environ['KAGGLE_KEY'] = kaggle_key

# kaggle.json
kpath = Path.home() / '.kaggle' / 'kaggle.json'
kpath.parent.mkdir(parents=True, exist_ok=True)
kpath.write_text(json.dumps({'username': kaggle_user, 'key': kaggle_key}))
kpath.chmod(0o600)

print(f'HF token: ...{hf_key[-8:]}')
print(f'Kaggle user: {kaggle_user}')

# ============ 2. GPU check ============
import torch
assert torch.cuda.is_available(), 'CUDA/GPU required (use Colab H100 or A100)'
d = torch.cuda.get_device_properties(0)
total_gb = d.total_memory / 1024**3
print(f'GPU: {d.name} {total_gb:.1f}GB')
assert total_gb >= 38, f'Need 40GB+ GPU, got {total_gb:.1f}GB'

# ============ 3. Install torch 2.5.1 + mamba-ssm + ML stack ============
print()
print('Installing dependencies (torch 2.5.1 + mamba-ssm + ML stack)...')
print('Expected: ~10 min (torch wheel download + install)')


def sh(cmd, timeout=900, check=True):
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    if check and r.returncode != 0:
        print(f'  FAIL: {" ".join(cmd[:6])}')
        print(f'  stderr: {r.stderr[-500:]}')
        raise RuntimeError('Command failed')
    return r


# Uninstall torch + Colab pre-installed torch.* packages (torch 2.11 ABI conflicts)
print('  Uninstalling existing torch + Colab pre-installed torchcodec/torchao/etc...')
for pkg in ['torch', 'torchvision', 'torchaudio',
            'torchcodec', 'torchao', 'torchdata', 'torchtune', 'torchsummary']:
    for i in range(5):
        r = sh([sys.executable, '-m', 'pip', 'uninstall', '-y', pkg], check=False)
        if 'Successfully uninstalled' not in r.stdout:
            break

# Install torch 2.5.1+cu124 via DIRECT wheel (bypass index-url resolution issues)
print('  Installing torch 2.5.1+cu124 (direct wheels)...')
for url in [
    'https://download.pytorch.org/whl/cu124/torch-2.5.1%2Bcu124-cp312-cp312-linux_x86_64.whl',
    'https://download.pytorch.org/whl/cu124/torchvision-0.20.1%2Bcu124-cp312-cp312-linux_x86_64.whl',
    'https://download.pytorch.org/whl/cu124/torchaudio-2.5.1%2Bcu124-cp312-cp312-linux_x86_64.whl',
]:
    sh([sys.executable, '-m', 'pip', 'install', '--force-reinstall', '--no-deps', url])

# torch runtime deps
sh([sys.executable, '-m', 'pip', 'install', '-q',
    'filelock', 'jinja2', 'networkx', 'fsspec', 'sympy>=1.13', 'typing-extensions'])

# Mamba-ssm + causal-conv1d (torch 2.5 ABI wheels)
print('  Installing mamba-ssm + causal-conv1d (torch 2.5 ABI wheels)...')
sh([sys.executable, '-m', 'pip', 'install', '--force-reinstall', '--no-deps',
    'https://github.com/state-spaces/mamba/releases/download/v2.2.4/mamba_ssm-2.2.4+cu12torch2.5cxx11abiFALSE-cp312-cp312-linux_x86_64.whl'])
sh([sys.executable, '-m', 'pip', 'install', '--force-reinstall', '--no-deps',
    'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.0.post8/causal_conv1d-1.5.0.post8+cu12torch2.5cxx11abiFALSE-cp312-cp312-linux_x86_64.whl'])

# ML stack
print('  Installing transformers/peft/trl/accelerate/datasets/bitsandbytes...')
sh([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.48,<4.58', 'peft>=0.14,<0.18', 'trl>=0.14,<0.26',
    'accelerate>=1.0,<2.0', 'datasets>=3.2,<5',
    'bitsandbytes', 'huggingface_hub', 'safetensors', 'einops',
    'sentencepiece', 'pandas', 'kagglehub', 'einx'])

# Unsloth --no-deps (no torch upgrade)
print('  Installing unsloth + unsloth_zoo (--no-deps)...')
sh([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
    'unsloth', 'unsloth_zoo', 'xformers', 'tyro', 'hf_transfer'], check=False)

# Verify via child process (child = fresh torch 2.5 import from disk)
print()
print('Verifying install via child process (torch 2.5 clean import)...')
r = subprocess.run([sys.executable, '-c', (
    "import torch, mamba_ssm; "
    "print(f'child: torch={torch.__version__} cuda={torch.version.cuda}'); "
    "print(f'child: mamba_ssm={mamba_ssm.__version__}'); "
    "from mamba_ssm.ops.triton.layernorm_gated import rmsnorm_fn; "
    "from unsloth import FastLanguageModel; "
    "print('child: ALL IMPORTS OK')"
)], capture_output=True, text=True, timeout=120)
print(r.stdout)
if r.returncode != 0:
    print('child stderr:', r.stderr[-500:])
    raise RuntimeError('Child process verify failed - deps broken')

# ============ 4. Download training script from GitHub ============
print()
print('Downloading training script from GitHub (colab_mega_v80.py)...')
SCRIPT_URL = 'https://raw.githubusercontent.com/FELIPEACASTRO/KG1-NVIDIA/claude/competent-shamir/scripts/colab_mega_v80.py'
SCRIPT_PATH = '/content/colab_mega_v80.py'

try:
    urllib.request.urlretrieve(SCRIPT_URL, SCRIPT_PATH)
    sz = os.path.getsize(SCRIPT_PATH)
    print(f'  Downloaded: {SCRIPT_PATH} ({sz/1024:.1f} KB)')
except Exception as e:
    print(f'  GitHub download failed: {e}')
    print('  Trying HF dataset repo fallback...')
    from huggingface_hub import hf_hub_download
    local = hf_hub_download(
        repo_id='felipesp1983/kg1-nemotron-training',
        filename='scripts/colab_mega_v80.py',
        repo_type='dataset',
        token=hf_key,
    )
    import shutil
    shutil.copy2(local, SCRIPT_PATH)
    print(f'  HF fallback OK: {SCRIPT_PATH} ({os.path.getsize(SCRIPT_PATH)/1024:.1f} KB)')

# ============ 5. Execute training in subprocess (child has clean torch 2.5) ============
print()
print('=' * 70)
print('Starting V80 training in child process (streams output live)')
print('Expected: ~3-4h (download if cold + train 245 steps + submit)')
print('=' * 70)
print()

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['PYTHONIOENCODING'] = 'utf-8'
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # anti-fragmentation
env['MAX_SEQ_LEN'] = '3072'  # p99 safe, H100 80GB fit, ~40s/step (vs 2.75min com 4096)

proc = subprocess.Popen(
    [sys.executable, '-u', SCRIPT_PATH],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in iter(proc.stdout.readline, ''):
    print(line, end='', flush=True)
proc.wait()

print()
print('=' * 70)
print(f'V80 MEGA DONE - return code {proc.returncode}')
if proc.returncode == 0:
    print('SUCCESS: training finished, submission made')
    print('Check Kaggle score: https://www.kaggle.com/competitions/nvidia-nemotron-model-reasoning-challenge/submissions')
else:
    print('FAILED: see output above for diagnosis')
print('=' * 70)


V80 MEGA CELL - dgxchen v7 EXACT, 1 cell, no restart
HF token: ...ifYYkxHG
Kaggle user: felipe1983
GPU: NVIDIA H100 80GB HBM3 79.2GB

Installing dependencies (torch 2.5.1 + mamba-ssm + ML stack)...
Expected: ~10 min (torch wheel download + install)
  Uninstalling existing torch + Colab pre-installed torchcodec/torchao/etc...
  Installing torch 2.5.1+cu124 (direct wheels)...
  Installing mamba-ssm + causal-conv1d (torch 2.5 ABI wheels)...
  Installing transformers/peft/trl/accelerate/datasets/bitsandbytes...
  Installing unsloth + unsloth_zoo (--no-deps)...

Verifying install via child process (torch 2.5 clean import)...
child: torch=2.5.1+cu124 cuda=12.4
child: mamba_ssm=2.2.4
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
child: ALL IMPORTS OK


  Downloaded: /content/colab_mega_v80.py (18.8 KB)

Starting V80 training in child process (streams output live)
Expected: ~3-4h (download if cold + tra

KeyboardInterrupt: 

In [2]:
# V3.1 PATCH - restart com paged_adamw_8bit (3-4x speedup real)
import subprocess, sys, os, urllib.request, time, torch, gc

print('=' * 70)
print('V3.1 PATCH - optim=paged_adamw_8bit (economiza 5GB VRAM)')
print('AdamW FP32 7GB -> AdamW 8bit 1.8GB -> Unsloth desliga offload -> 3-4x speedup')
print('Expected ETA: 19h -> ~2.5h')
print('=' * 70)

# 1. Kill V3 subprocess
subprocess.run(['pkill', '-9', '-f', 'colab_mega_v80.py'], capture_output=True)
time.sleep(3)
print('V3 subprocess killed')

# 2. Clear GPU
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print(f'GPU free: {torch.cuda.mem_get_info()[0]/1024**3:.1f}GB')

# 3. Download V3.1 script (commit 1c73ed5)
print('\nDownloading V3.1 script (paged_adamw_8bit fix)...')
url = f'https://raw.githubusercontent.com/FELIPEACASTRO/KG1-NVIDIA/claude/competent-shamir/scripts/colab_mega_v80.py?t={int(time.time())}'
urllib.request.urlretrieve(url, '/content/colab_mega_v80.py')

# Verify fix
with open('/content/colab_mega_v80.py') as f:
    content = f.read()
assert 'paged_adamw_8bit' in content, 'V3.1 fix missing'
print('V3.1 fix verified (paged_adamw_8bit present)')

# 4. Run subprocess
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['PYTHONIOENCODING'] = 'utf-8'
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
env['MAX_SEQ_LEN'] = '3072'

print('\n' + '=' * 70)
print('V3.1 training starting (ETA ~2.5h)')
print('Cache: install + model 50GB + dataset (all preserved)')
print('=' * 70)
print()

proc = subprocess.Popen(
    [sys.executable, '-u', '/content/colab_mega_v80.py'],
    env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in iter(proc.stdout.readline, ''):
    print(line, end='', flush=True)
proc.wait()

print(f'\nV3.1 DONE - rc={proc.returncode}')

V3.1 PATCH - optim=paged_adamw_8bit (economiza 5GB VRAM)
AdamW FP32 7GB -> AdamW 8bit 1.8GB -> Unsloth desliga offload -> 3-4x speedup
Expected ETA: 19h -> ~2.5h
V3 subprocess killed
GPU free: 78.7GB

V3.1 fix verified (paged_adamw_8bit present)

V3.1 training starting (ETA ~2.5h)
Cache: install + model 50GB + dataset (all preserved)


[CHILD] torch=2.5.1+cu124  cuda=12.4
[CHILD] GPU: NVIDIA H100 80GB HBM3 79.2GB
[CHILD] mamba_ssm=2.2.4

STEP 2/7: Download dgxchen dataset (problem_ids_matched.csv)
Dataset cached: 46.1MB
Dataset path: /content/kg1_data/problem_ids_matched.csv
Rows: 7830
Columns: ['id', 'prompt', 'answer', 'type', 'generated_cot']
Type distribution:
  bit_manipulation: 1754
  cipher: 1656
  unit_conversion: 1070
  gravity: 1055
  numeral: 730
  equation_numeric_deduce: 658
  cryptarithm_deduce: 627
  cryptarithm_guess: 154
  equation_numeric_guess: 126

STEP 3/7: Download Nemotron-3-Nano-30B-A3B-BF16 base model
Model cached at: /root/.cache/kagglehub/models/metric/nemotr

KeyboardInterrupt: 

In [5]:
# Automação já commitada (f3f282a):
python scripts/kg1_v80_to_v81_orchestrator.py \
    --adapter-dir /content/kg1_adapter_v80 \
    --output-dir /content/kg1_v81_output \
    --submit


SyntaxError: invalid syntax (3934712871.py, line 2)

In [9]:
# =============================================================================
# V80 MEGA CELL FINAL - 14 fixes inline - dgxchen v7 EXACT replica (0.85 LB target)
# 1 célula, zero restart, zero dependência de GitHub
# Expected: ~3h H100 80GB -> score 0.84-0.85
# =============================================================================
import subprocess, sys, os, json, gc, time, urllib.request
from pathlib import Path

print('=' * 70)
print('V80 MEGA FINAL - 14 fixes consolidados')
print('=' * 70)

# ===== PART 1: SECRETS =====
try:
    from google.colab import userdata
    hf_key = userdata.get('HF_KEY')
    kaggle_user = userdata.get('KAGGLE_USERNAME')
    kaggle_key = userdata.get('KAGGLE_KEY')
except Exception:
    hf_key = os.environ.get('HF_KEY') or os.environ.get('HF_TOKEN')
    kaggle_user = os.environ.get('KAGGLE_USERNAME')
    kaggle_key = os.environ.get('KAGGLE_KEY')

assert hf_key, 'HF_KEY missing - add in Colab Secrets (cadeado)'
assert kaggle_user and kaggle_key, 'KAGGLE_USERNAME / KAGGLE_KEY missing'

os.environ['HF_TOKEN'] = hf_key
os.environ['HF_KEY'] = hf_key
os.environ['KAGGLE_USERNAME'] = kaggle_user
os.environ['KAGGLE_KEY'] = kaggle_key

kpath = Path.home() / '.kaggle' / 'kaggle.json'
kpath.parent.mkdir(parents=True, exist_ok=True)
kpath.write_text(json.dumps({'username': kaggle_user, 'key': kaggle_key}))
kpath.chmod(0o600)

print(f'HF token: ...{hf_key[-8:]}')
print(f'Kaggle user: {kaggle_user}')

# ===== PART 2: GPU CHECK =====
import torch
assert torch.cuda.is_available(), 'GPU required'
d = torch.cuda.get_device_properties(0)
total_gb = d.total_memory / 1024**3
print(f'GPU: {d.name} {total_gb:.1f}GB')
assert total_gb >= 38, f'Need 40GB+ GPU'

# ===== PART 3: INSTALL DEPS =====
print()
print('Installing dependencies (torch 2.5.1 + mamba-ssm + ML stack)...')

def sh(cmd, timeout=900, check=True):
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    if check and r.returncode != 0:
        print(f'  FAIL: {" ".join(cmd[:6])}')
        print(r.stderr[-500:])
        raise RuntimeError('Command failed')
    return r

# FIX 11: Uninstall Colab pre-installed torch + torchcodec/torchao/etc (torch 2.11 ABI conflict)
print('  Uninstalling torch + Colab pre-installed torchcodec/torchao/torchdata...')
for pkg in ['torch', 'torchvision', 'torchaudio',
            'torchcodec', 'torchao', 'torchdata', 'torchtune', 'torchsummary']:
    for _ in range(5):
        r = sh([sys.executable, '-m', 'pip', 'uninstall', '-y', pkg], check=False)
        if 'Successfully uninstalled' not in r.stdout:
            break

# FIX 9: Install torch 2.5.1+cu124 via DIRECT WHEEL URLs (bypass --index-url resolution issues)
print('  Installing torch 2.5.1+cu124 (direct wheels)...')
for url in [
    'https://download.pytorch.org/whl/cu124/torch-2.5.1%2Bcu124-cp312-cp312-linux_x86_64.whl',
    'https://download.pytorch.org/whl/cu124/torchvision-0.20.1%2Bcu124-cp312-cp312-linux_x86_64.whl',
    'https://download.pytorch.org/whl/cu124/torchaudio-2.5.1%2Bcu124-cp312-cp312-linux_x86_64.whl',
]:
    sh([sys.executable, '-m', 'pip', 'install', '--force-reinstall', '--no-deps', url])

sh([sys.executable, '-m', 'pip', 'install', '-q',
    'filelock', 'jinja2', 'networkx', 'fsspec', 'sympy>=1.13', 'typing-extensions'])

# FIX 8: Install mamba-ssm + causal-conv1d wheels (torch 2.5 ABI, cp312+cu12)
print('  Installing mamba-ssm + causal-conv1d (torch 2.5 ABI wheels)...')
sh([sys.executable, '-m', 'pip', 'install', '--force-reinstall', '--no-deps',
    'https://github.com/state-spaces/mamba/releases/download/v2.2.4/'
    'mamba_ssm-2.2.4+cu12torch2.5cxx11abiFALSE-cp312-cp312-linux_x86_64.whl'])
sh([sys.executable, '-m', 'pip', 'install', '--force-reinstall', '--no-deps',
    'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.0.post8/'
    'causal_conv1d-1.5.0.post8+cu12torch2.5cxx11abiFALSE-cp312-cp312-linux_x86_64.whl'])

# ML stack (without upgrading torch)
print('  Installing transformers/peft/trl/accelerate/datasets/bitsandbytes...')
sh([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.48,<4.58', 'peft>=0.14,<0.18', 'trl>=0.14,<0.26',
    'accelerate>=1.0,<2.0', 'datasets>=3.2,<5',
    'bitsandbytes', 'huggingface_hub', 'safetensors', 'einops',
    'sentencepiece', 'pandas', 'kagglehub', 'einx'])

# FIX 11: Unsloth --no-deps (previne upgrade accidental torch para 2.11)
print('  Installing unsloth + unsloth_zoo (--no-deps)...')
sh([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
    'unsloth', 'unsloth_zoo', 'xformers', 'tyro', 'hf_transfer'], check=False)

# Verify child process sees torch 2.5 clean
print()
print('Verifying child process (torch 2.5 clean import)...')
r = subprocess.run([sys.executable, '-c', (
    "import torch, mamba_ssm; "
    "print(f'child: torch={torch.__version__} cuda={torch.version.cuda}'); "
    "print(f'child: mamba_ssm={mamba_ssm.__version__}'); "
    "from mamba_ssm.ops.triton.layernorm_gated import rmsnorm_fn; "
    "from unsloth import FastLanguageModel; "
    "print('child: ALL IMPORTS OK')"
)], capture_output=True, text=True, timeout=180)
print(r.stdout)
if r.returncode != 0:
    print('child stderr:', r.stderr[-500:])
    raise RuntimeError('Deps broken')

# ===== PART 4: WRITE TRAINING SCRIPT INLINE =====
TRAIN_SCRIPT = r'''#!/usr/bin/env python3
"""V80 dgxchen v7 EXACT - training script with all 14 fixes."""
import sys, os, gc, re, math, time, json, random, shutil, zipfile, datetime, subprocess
from pathlib import Path
from collections import defaultdict, deque

for s in (sys.stdout, sys.stderr):
    if hasattr(s, "reconfigure"): s.reconfigure(encoding="utf-8", errors="replace")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TQDM_DISABLE", "1")

import torch
print(f"\n[CHILD] torch={torch.__version__}  cuda={torch.version.cuda}")
assert torch.__version__.startswith("2.5"), f"Need torch 2.5, got {torch.__version__}"
d = torch.cuda.get_device_properties(0)
print(f"[CHILD] GPU: {d.name} {d.total_memory/1024**3:.1f}GB")

import mamba_ssm
from mamba_ssm.ops.triton.layernorm_gated import rmsnorm_fn
from mamba_ssm.ops.selective_scan_interface import selective_scan_fn
print(f"[CHILD] mamba_ssm={mamba_ssm.__version__}")

SEED = 42
random.seed(SEED); import numpy as np; np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HF_KEY")
MAX_SEQ_LEN = int(os.environ.get("MAX_SEQ_LEN", 3072))  # FIX 12: p99 safe
print(f"[CHILD] MAX_SEQ_LEN={MAX_SEQ_LEN}")

# ===== STEP 2: Dataset =====
print("\n" + "=" * 70)
print("STEP 2/7: Download dgxchen dataset (FIX 1: problem_ids_matched.csv)")
print("=" * 70)

DATA_DIR = Path("/content/kg1_data")
DATA_DIR.mkdir(exist_ok=True)
target_csv = DATA_DIR / "problem_ids_matched.csv"

if target_csv.exists() and target_csv.stat().st_size > 40_000_000:
    print(f"Dataset cached: {target_csv.stat().st_size/(1024**2):.1f}MB")
else:
    r = subprocess.run(
        ["kaggle", "datasets", "download", "-d", "dgxchen/nemotron-cot-tong",
         "-p", str(DATA_DIR), "--unzip"],
        capture_output=True, text=True, timeout=300)
    print(r.stdout)
    if r.returncode != 0:
        raise RuntimeError(f"Kaggle download failed: {r.stderr[-500:]}")

import pandas as pd
df = pd.read_csv(target_csv)
print(f"Rows: {len(df)}  Columns: {list(df.columns)}")
for t, n in df["type"].value_counts().items():
    print(f"  {t}: {n}")

# ===== STEP 3: Base model =====
print("\n" + "=" * 70)
print("STEP 3/7: Nemotron-3-Nano-30B-A3B-BF16 base model")
print("=" * 70)

import kagglehub
MODEL_CACHE = "/root/.cache/kagglehub/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"
if Path(MODEL_CACHE).exists() and len(list(Path(MODEL_CACHE).iterdir())) > 5:
    MODEL_PATH = MODEL_CACHE
    print(f"Model cached: {MODEL_PATH}")
else:
    print("Downloading base model ~60GB (~50min first time)...")
    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    print(f"Model path: {MODEL_PATH}")

# ===== STEP 4: Model + LoRA =====
print("\n" + "=" * 70)
print("STEP 4/7: Load + LoRA (FIX 2: attn=eager, FIX 3: 8 targets NO lm_head)")
print("=" * 70)

from unsloth import FastLanguageModel
print(f"Loading via Unsloth (attn=eager, max_seq_len={MAX_SEQ_LEN})...")
t_load = time.time()
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=False, load_in_8bit=False, full_finetuning=False,
    trust_remote_code=True, unsloth_force_compile=False,
    attn_implementation="eager",  # FIX 2
    dtype=torch.bfloat16,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Model loaded in {time.time()-t_load:.1f}s")

# FIX 3: 8 targets NO lm_head (dgxchen v7)
target_modules = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "in_proj", "out_proj", "up_proj", "down_proj",
]
print(f"LoRA r=32 alpha=32 dropout=0.0, targets ({len(target_modules)}, NO lm_head): {target_modules}")
model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=32, lora_dropout=0.0,
    target_modules=target_modules, bias="none",
    use_gradient_checkpointing="unsloth", random_state=SEED,
)
model.print_trainable_parameters()
free_gb = torch.cuda.mem_get_info()[0] / 1024**3
used_gb = (torch.cuda.mem_get_info()[1] - torch.cuda.mem_get_info()[0]) / 1024**3
print(f"After LoRA: used={used_gb:.1f}GB free={free_gb:.1f}GB")

# ===== STEP 5: SFT records =====
print("\n" + "=" * 70)
print("STEP 5/7: Build SFT records + stratified sampler")
print("=" * 70)

from datasets import Dataset as HFDataset
from torch.utils.data import DataLoader, Sampler
from transformers import TrainerCallback
from trl import SFTTrainer, SFTConfig

PROMPT_SUFFIX = "\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`"
train_df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
records, record_types = [], []
for _, row in train_df.iterrows():
    cot = str(row.get("generated_cot", ""))
    if not cot or cot == "nan" or len(cot.strip()) < 5: continue
    cot_clean = re.sub(r"\\boxed\{[^}]*\}", "", cot).rstrip()
    user_content = str(row["prompt"]) + PROMPT_SUFFIX
    assistant_content = cot_clean + f"\n</think>\n\\boxed{{{row['answer']}}}"
    records.append({"messages": [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": assistant_content},
    ]})
    record_types.append(str(row.get("type", "unknown")))
print(f"SFT records: {len(records)}")
dataset = HFDataset.from_list(records)

# FIX 7: formatting_func com conversation wrap (dgxchen v7)
def formatting_prompts_func(example):
    messages = example["messages"]
    conversations = [messages] if messages and isinstance(messages[0], dict) else messages
    texts = []
    for conv in conversations:
        try:
            t = tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=False, enable_thinking=True)
        except TypeError:
            t = tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=False)
        texts.append(t)
    return texts

def build_stratified_order(labels, batch_size, seed):
    by_label = defaultdict(list)
    for i, l in enumerate(labels): by_label[l].append(i)
    rng = random.Random(seed)
    for v in by_label.values(): rng.shuffle(v)
    n_batches = max(1, math.ceil(len(labels) / batch_size))
    batches = [[] for _ in range(n_batches)]
    order = list(range(n_batches)); rng.shuffle(order)
    assigned = 0
    for label in sorted(by_label.keys()):
        for idx in by_label[label]:
            batches[order[assigned % n_batches]].append(idx)
            assigned += 1
    return [i for b in batches for i in b]

class OrderSampler(Sampler):
    def __init__(self, order): self.order = list(order)
    def __iter__(self): return iter(self.order)
    def __len__(self): return len(self.order)

class StratSFTTrainer(SFTTrainer):
    def __init__(self, *args, stratified_order=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.stratified_order = stratified_order
    def get_train_dataloader(self):
        if self.stratified_order is None: return super().get_train_dataloader()
        dk = {"batch_size": self.args.per_device_train_batch_size,
              "sampler": OrderSampler(self.stratified_order),
              "collate_fn": self.data_collator,
              "num_workers": self.args.dataloader_num_workers,
              "pin_memory": self.args.dataloader_pin_memory,
              "persistent_workers": self.args.dataloader_persistent_workers,
              "drop_last": self.args.dataloader_drop_last}
        if self.args.dataloader_num_workers > 0:
            dk["prefetch_factor"] = self.args.dataloader_prefetch_factor
        return DataLoader(self.train_dataset, **dk)

class HealthGateCallback(TrainerCallback):
    def __init__(self):
        self.loss_hist = deque(maxlen=10); self.grad_hist = deque(maxlen=5)
        self.min_loss = float("inf"); self.steps_no_improve = 0
        self.high_grad = 0; self.t0 = None
    def on_train_begin(self, args, state, control, **kw):
        self.t0 = time.time()
        print("=" * 70 + "\nV80 HEALTH GATES ATIVOS\n" + "=" * 70)
    def on_log(self, args, state, control, logs=None, **kw):
        if not logs: return
        step = int(state.global_step)
        loss = logs.get("loss"); grad = logs.get("grad_norm")
        lr = logs.get("learning_rate"); epoch = logs.get("epoch", 0.0)
        if loss is not None and isinstance(loss, float):
            if math.isnan(loss) or math.isinf(loss):
                print(f"!!! NaN/Inf step {step} ABORT"); control.should_training_stop = True; return
            if loss > 30.0 and step > 10:
                print(f"!!! loss explosion {loss:.3f} step {step} ABORT"); control.should_training_stop = True; return
            self.loss_hist.append(float(loss))
            if float(loss) < self.min_loss: self.min_loss = float(loss); self.steps_no_improve = 0
            else: self.steps_no_improve += 5
        if grad is not None:
            self.grad_hist.append(float(grad))
            self.high_grad = self.high_grad + 1 if float(grad) > 50.0 else 0
            if self.high_grad >= 3: print(f"!!! grad>50 3x (no clipping)")
        free = torch.cuda.mem_get_info()[0] / 1024**3
        total = torch.cuda.mem_get_info()[1] / 1024**3
        used = total - free; peak = torch.cuda.max_memory_allocated() / 1024**3
        if free < 2.0: print(f"!!! VRAM free={free:.1f}GB OOM risk")
        el = time.time() - self.t0 if self.t0 else 0
        total_s = state.max_steps if state.max_steps else 1
        eta = (el / max(step, 1)) * (total_s - step) if step > 0 else 0
        avg_l = sum(self.loss_hist)/len(self.loss_hist) if self.loss_hist else 0
        avg_g = sum(self.grad_hist)/len(self.grad_hist) if self.grad_hist else 0
        _l = float(loss) if loss is not None else 0.0
        _g = float(grad) if grad is not None else 0.0
        _r = float(lr) if lr is not None else 0.0
        print(f"[step {step:4d}/{total_s:4d} ep{epoch:.2f} {step/max(total_s,1)*100:5.1f}%] "
              f"loss={_l:.4f} avg10={avg_l:.4f} grad={_g:.3f} avg5={avg_g:.3f} "
              f"lr={_r:.2e} vram={used:.1f}/{peak:.1f}/{free:.1f}GB "
              f"elapsed={int(el//60)}m ETA={int(eta//60)}m")
    def on_train_end(self, args, state, control, **kw):
        el = time.time() - self.t0 if self.t0 else 0
        print(f"=" * 70 + f"\nTRAIN DONE elapsed={el/60:.1f}min min_loss={self.min_loss:.4f}\n" + "=" * 70)

# ===== STEP 6: Training =====
print("\n" + "=" * 70)
print("STEP 6/7: Train 1 epoch (dgxchen v7 EXACT + FIX 14: paged_adamw_8bit)")
print("=" * 70)

OUT_DIR = "/content/kg1_out/sft_v80"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

training_args = SFTConfig(
    output_dir=OUT_DIR,
    num_train_epochs=1,                          # FIX 4
    per_device_train_batch_size=1,
    gradient_accumulation_steps=32,
    learning_rate=2e-4,
    lr_scheduler_type="linear",
    warmup_steps=0,
    max_length=MAX_SEQ_LEN,                      # FIX 12: 3072
    adam_beta1=0.9, adam_beta2=0.95, adam_epsilon=1e-8,
    weight_decay=0.0,
    max_grad_norm=1e9,                           # FIX 5
    optim="paged_adamw_8bit",                    # FIX 14: 8bit optimizer (-5GB VRAM)
    logging_steps=5, logging_first_step=True,
    save_strategy="steps", save_steps=50, save_total_limit=3,
    bf16=True,
    gradient_checkpointing=True,                 # FIX 6
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=0,                    # FIX 10
    remove_unused_columns=False,
    seed=SEED, report_to="none", packing=False,
)

order = build_stratified_order(record_types, 32, SEED)
print(f"Eff batch: 32  Total optim steps: {math.ceil(len(record_types)/32)}")

trainer = StratSFTTrainer(
    model=model, args=training_args, train_dataset=dataset,
    processing_class=tokenizer,
    formatting_func=formatting_prompts_func,     # FIX 7
    stratified_order=order,
    callbacks=[HealthGateCallback()],
)

print("Starting V80 SFT...")
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
trainer.train()
print(f"Training done: {(time.time()-t0)/60:.1f} min")
print(f"Peak VRAM: {torch.cuda.max_memory_allocated()/1024**3:.2f}GB")

# Save adapter
ADAPTER_DIR = "/content/kg1_adapter_v80"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved: {ADAPTER_DIR}")

# ===== STEP 7: Submit =====
print("\n" + "=" * 70)
print("STEP 7/7: Submission.zip + HF upload + Kaggle submit")
print("=" * 70)

BASE = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
SUB_DIR = "/content/kg1_out/submission_v80"
Path(SUB_DIR).mkdir(parents=True, exist_ok=True)
required = ["adapter_config.json", "adapter_model.safetensors"]
for fn in required:
    shutil.copy2(Path(ADAPTER_DIR)/fn, Path(SUB_DIR)/fn)
    print(f"Copied {fn} ({(Path(SUB_DIR)/fn).stat().st_size/1024**2:.1f}MB)")

cfg_path = Path(SUB_DIR) / "adapter_config.json"
with open(cfg_path) as f: cfg = json.load(f)
cfg["base_model_name_or_path"] = BASE
cfg["inference_mode"] = True
cfg["lora_dropout"] = 0.0
with open(cfg_path, "w") as f: json.dump(cfg, f, indent=2)
tm = cfg.get("target_modules", [])
print(f"adapter targets: {len(tm) if isinstance(tm, list) else 'parameter list'}, lm_head present: {'lm_head' in tm if isinstance(tm, list) else 'N/A'}")

zip_path = "/content/kg1_out/submission_v80.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fn in required:
        zf.write(Path(SUB_DIR)/fn, arcname=fn)
print(f"zip: {os.path.getsize(zip_path)/1024**2:.1f}MB")

# HF upload
try:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    REPO = "felipesp1983/kg1-nemotron-lora-v80-final"
    api.create_repo(REPO, private=True, exist_ok=True)
    api.upload_folder(repo_id=REPO, folder_path=SUB_DIR,
                      allow_patterns=["adapter_*"], token=HF_TOKEN)
    print(f"HF: https://huggingface.co/{REPO}")
except Exception as e:
    print(f"HF upload failed (non-fatal): {e}")

# Kaggle submit (slot check)
try:
    rc = subprocess.run(["kaggle", "competitions", "submissions",
                         "-c", "nvidia-nemotron-model-reasoning-challenge", "--csv"],
                        capture_output=True, text=True, timeout=60)
    if rc.returncode == 0:
        from io import StringIO; import csv as _csv
        today = datetime.datetime.now().strftime("%Y-%m-%d")
        cnt = sum(1 for r in _csv.DictReader(StringIO(rc.stdout)) if r.get("date","").startswith(today))
        print(f"Submissions today: {cnt}/5")
        if cnt < 5:
            msg = f"V80 FINAL dgxchen v7 EXACT {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}"
            r = subprocess.run(["kaggle", "competitions", "submit",
                                "-c", "nvidia-nemotron-model-reasoning-challenge",
                                "-f", zip_path, "-m", msg],
                               capture_output=True, text=True, timeout=600)
            print(f"Submit rc={r.returncode}\n{r.stdout[-400:]}")
        else:
            print(f"Slot full. Manual: kaggle competitions submit -c nvidia-nemotron-model-reasoning-challenge -f {zip_path} -m V80")
except Exception as e:
    print(f"Submit fail: {e}")

print("\n" + "=" * 70)
print("V80 FINAL DONE - check score at")
print("https://www.kaggle.com/competitions/nvidia-nemotron-model-reasoning-challenge/submissions")
print("=" * 70)
'''

with open('/content/v80_final.py', 'w', encoding='utf-8') as f:
    f.write(TRAIN_SCRIPT)
print(f'Training script written to /content/v80_final.py ({os.path.getsize("/content/v80_final.py")/1024:.1f} KB)')

# ===== PART 5: RUN TRAINING =====
print()
print('=' * 70)
print('Starting V80 FINAL training (ETA ~2.5-3h)')
print('All 14 fixes applied:')
print('  1. Dataset problem_ids_matched.csv')
print('  2. attn_implementation=eager')
print('  3. LoRA 8 targets NO lm_head')
print('  4. num_train_epochs=1')
print('  5. max_grad_norm=1e9')
print('  6. gradient_checkpointing=True + use_reentrant=False')
print('  7. formatting_func trainer + conversation wrap')
print('  8. mamba-ssm + causal-conv1d wheels')
print('  9. torch 2.5.1 via wheels diretos')
print(' 10. dataloader_num_workers=0')
print(' 11. Unsloth --no-deps + uninstall torchcodec')
print(' 12. MAX_SEQ_LEN=3072 (p99 safe)')
print(' 13. PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True')
print(' 14. optim=paged_adamw_8bit (-5GB VRAM, no offload, 3-4x speedup)')
print('=' * 70)
print()

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['PYTHONIOENCODING'] = 'utf-8'
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # FIX 13
env['MAX_SEQ_LEN'] = '3072'  # FIX 12

proc = subprocess.Popen(
    [sys.executable, '-u', '/content/v80_final.py'],
    env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in iter(proc.stdout.readline, ''):
    print(line, end='', flush=True)
proc.wait()

print()
print('=' * 70)
print(f'V80 FINAL DONE - rc={proc.returncode}')
print('=' * 70)

V80 MEGA FINAL - 14 fixes consolidados
HF token: ...ifYYkxHG
Kaggle user: felipe1983
GPU: NVIDIA H100 80GB HBM3 79.2GB

Installing dependencies (torch 2.5.1 + mamba-ssm + ML stack)...
  Uninstalling torch + Colab pre-installed torchcodec/torchao/torchdata...
  Installing torch 2.5.1+cu124 (direct wheels)...
  Installing mamba-ssm + causal-conv1d (torch 2.5 ABI wheels)...
  Installing transformers/peft/trl/accelerate/datasets/bitsandbytes...
  Installing unsloth + unsloth_zoo (--no-deps)...

Verifying child process (torch 2.5 clean import)...
child: torch=2.5.1+cu124 cuda=12.4
child: mamba_ssm=2.2.4
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
child: ALL IMPORTS OK

Training script written to /content/v80_final.py (14.9 KB)

Starting V80 FINAL training (ETA ~2.5-3h)
All 14 fixes applied:
  1. Dataset problem_ids_matched.csv
  2. attn_implementation=eager
  3. LoRA 8 targets NO lm_head
  4. num_t

KeyboardInterrupt: 

In [10]:
import os, subprocess, json, zipfile
from pathlib import Path

# Credenciais
os.environ['KAGGLE_USERNAME'] = 'felipe1983'
os.environ['KAGGLE_KEY'] = '93dbcf741dba9085eded2cdbe2fc0cab'

# Save kaggle.json
kpath = Path.home() / '.kaggle' / 'kaggle.json'
kpath.parent.mkdir(parents=True, exist_ok=True)
kpath.write_text(json.dumps({'username': 'felipe1983', 'key': '93dbcf741dba9085eded2cdbe2fc0cab'}))
kpath.chmod(0o600)

# 1. Checar / criar submission.zip
zip_path = Path('/content/kg1_out/submission_v80.zip')
adapter_dir = Path('/content/kg1_adapter_v80')

if not zip_path.exists():
    print('Creating submission.zip from adapter...')

    # Patch adapter_config.json
    cfg_path = adapter_dir / 'adapter_config.json'
    with open(cfg_path) as f:
        cfg = json.load(f)
    cfg['base_model_name_or_path'] = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'
    cfg['inference_mode'] = True
    cfg['lora_dropout'] = 0.0
    with open(cfg_path, 'w') as f:
        json.dump(cfg, f, indent=2)

    tm = cfg.get('target_modules', [])
    print(f"target_modules ({len(tm) if isinstance(tm, list) else 'params'}): {tm}")
    print(f"lm_head present: {'lm_head' in tm if isinstance(tm, list) else 'N/A'}")

    # Build zip
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for fn in ['adapter_config.json', 'adapter_model.safetensors']:
            zf.write(adapter_dir / fn, arcname=fn)
    print(f'submission.zip created: {zip_path.stat().st_size / 1024**2:.1f} MB')
else:
    print(f'submission.zip já existe: {zip_path.stat().st_size / 1024**2:.1f} MB')

# 2. Verifica slots hoje
print('\n=== Current submissions today ===')
r = subprocess.run(['kaggle', 'competitions', 'submissions',
                    '-c', 'nvidia-nemotron-model-reasoning-challenge', '--csv'],
                   capture_output=True, text=True, timeout=60)
print(r.stdout[:500])

# 3. SUBMIT
print('\n=== SUBMITTING V80 ===')
r = subprocess.run(['kaggle', 'competitions', 'submit',
                    '-c', 'nvidia-nemotron-model-reasoning-challenge',
                    '-f', str(zip_path),
                    '-m', 'V80 FINAL dgxchen v7 EXACT replica - min_loss=1.4275'],
                   capture_output=True, text=True, timeout=600)
print(f'rc={r.returncode}')
print(r.stdout)
if r.stderr:
    print(f'stderr: {r.stderr[:500]}')

# 4. Confirma
print('\n=== Post-submit verification ===')
import time
time.sleep(5)
r = subprocess.run(['kaggle', 'competitions', 'submissions',
                    '-c', 'nvidia-nemotron-model-reasoning-challenge', '--csv'],
                   capture_output=True, text=True, timeout=60)
print(r.stdout[:600])

print('\n=== DONE ===')
print('Score em: https://www.kaggle.com/competitions/nvidia-nemotron-model-reasoning-challenge/submissions')

submission.zip já existe: 3093.1 MB

=== Current submissions today ===
fileName,date,description,status,publicScore,privateScore
submission.zip,2026-04-20 12:34:51.980000,v18 stripped attempt3,SubmissionStatus.COMPLETE,0.50,
submission.zip,2026-04-20 12:33:08.957000,v18 stripped attempt2,SubmissionStatus.COMPLETE,0.50,
submission.zip,2026-04-20 12:31:26.853000,v18 stripped attempt1,SubmissionStatus.COMPLETE,0.50,
submission.zip,2026-04-20 12:30:08.120000,v18 stripped submission.zip,SubmissionStatus.COMPLETE,0.52,
submission.zip,2026-04-17 12:45:58.347000,v87 checkp

=== SUBMITTING V80 ===
rc=1
400 Client Error: Bad Request for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/CreateSubmission

stderr: 
  0%|          | 0.00/3.02G [00:00<?, ?B/s]
  0%|          | 976k/3.02G [00:00<43:44, 1.24MB/s]
  0%|          | 1.33M/3.02G [00:01<41:26, 1.30MB/s]
  0%|          | 1.70M/3.02G [00:01<36:01, 1.50MB/s]
  0%|          | 2.75M/3.02G [00:01<20:54, 2.58MB/s]
  0%|          | 

In [11]:
import os, json, shutil, zipfile, subprocess, time
from pathlib import Path
from safetensors import safe_open
from safetensors.torch import save_file

os.environ['KAGGLE_USERNAME'] = 'felipe1983'
os.environ['KAGGLE_KEY'] = '93dbcf741dba9085eded2cdbe2fc0cab'

SRC = Path('/content/kg1_adapter_v80')
DST = Path('/content/kg1_adapter_v80_stripped')
DST.mkdir(parents=True, exist_ok=True)

# === 1. Analyze safetensors keys ===
print('=== Analyzing adapter_model.safetensors ===')
with safe_open(SRC / 'adapter_model.safetensors', framework='pt') as f:
    all_keys = list(f.keys())

expert_keys = [k for k in all_keys if 'mlp.experts' in k]
base_keys = [k for k in all_keys if 'mlp.experts' not in k]
print(f'Total keys: {len(all_keys)}')
print(f'Expert LoRA keys (will REMOVE): {len(expert_keys)}')
print(f'Base LoRA keys (will KEEP): {len(base_keys)}')
print(f'Sample base keys: {base_keys[:5]}')
print(f'Sample expert keys: {expert_keys[:3]}')

# === 2. Strip MoE experts, keep only base 8 targets ===
print('\n=== Stripping MoE experts (keeping 8 base targets only) ===')
kept = {}
with safe_open(SRC / 'adapter_model.safetensors', framework='pt') as f:
    for k in base_keys:
        kept[k] = f.get_tensor(k)

save_file(kept, str(DST / 'adapter_model.safetensors'))

# === 3. Patch adapter_config.json ===
with open(SRC / 'adapter_config.json') as f:
    cfg = json.load(f)

# Force 8 targets (remove any MoE references)
cfg['target_modules'] = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                          'in_proj', 'out_proj', 'up_proj', 'down_proj']
cfg['base_model_name_or_path'] = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'
cfg['inference_mode'] = True
cfg['lora_dropout'] = 0.0
# Remove any MoE-specific fields
for k in ['target_parameters', 'layers_to_transform', 'layers_pattern']:
    cfg.pop(k, None)

with open(DST / 'adapter_config.json', 'w') as f:
    json.dump(cfg, f, indent=2)

print(f'Patched adapter_config.json targets: {cfg["target_modules"]}')

# === 4. Size comparison ===
orig_mb = (SRC / 'adapter_model.safetensors').stat().st_size / 1024**2
strip_mb = (DST / 'adapter_model.safetensors').stat().st_size / 1024**2
print(f'\nOriginal: {orig_mb:.1f} MB')
print(f'Stripped: {strip_mb:.1f} MB (kept {strip_mb/orig_mb*100:.1f}%)')

# === 5. Build zip ===
zip_path = Path('/content/kg1_out/submission_v80_stripped.zip')
zip_path.parent.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fn in ['adapter_config.json', 'adapter_model.safetensors']:
        zf.write(DST / fn, arcname=fn)

zip_mb = zip_path.stat().st_size / 1024**2
print(f'\nZip: {zip_mb:.1f} MB')

if zip_mb > 500:
    print(f'!!! STILL > 500 MB. Cannot submit.')
else:
    print('Zip OK (<500MB). Ready to submit.')

    # === 6. SUBMIT ===
    print('\n=== Submitting V80 stripped ===')
    r = subprocess.run(
        ['kaggle', 'competitions', 'submit',
         '-c', 'nvidia-nemotron-model-reasoning-challenge',
         '-f', str(zip_path),
         '-m', f'V80 FINAL stripped MoE - 8 targets only - min_loss=1.4275'],
        capture_output=True, text=True, timeout=600
    )
    print(f'rc={r.returncode}')
    print(r.stdout)
    if r.stderr:
        print(f'stderr: {r.stderr[:500]}')

    # === 7. Verify ===
    print('\n=== Post-submit verification (wait 5s) ===')
    time.sleep(5)
    r = subprocess.run(['kaggle', 'competitions', 'submissions',
                        '-c', 'nvidia-nemotron-model-reasoning-challenge', '--csv'],
                       capture_output=True, text=True, timeout=60)
    print(r.stdout[:800])

print('\n=== DONE ===')

=== Analyzing adapter_model.safetensors ===
Total keys: 12008
Expert LoRA keys (will REMOVE): 0
Base LoRA keys (will KEEP): 12008
Sample base keys: ['base_model.model.backbone.layers.0.mixer.in_proj.lora_A.weight', 'base_model.model.backbone.layers.0.mixer.in_proj.lora_B.weight', 'base_model.model.backbone.layers.0.mixer.out_proj.lora_A.weight', 'base_model.model.backbone.layers.0.mixer.out_proj.lora_B.weight', 'base_model.model.backbone.layers.1.mixer.experts.0.down_proj.lora_A.weight']
Sample expert keys: []

=== Stripping MoE experts (keeping 8 base targets only) ===
Patched adapter_config.json targets: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'in_proj', 'out_proj', 'up_proj', 'down_proj']

Original: 3373.4 MB
Stripped: 3373.4 MB (kept 100.0%)

Zip: 3093.1 MB
!!! STILL > 500 MB. Cannot submit.

=== DONE ===


In [12]:
import os, json, shutil, zipfile, subprocess, time
from pathlib import Path
from safetensors import safe_open
from safetensors.torch import save_file

os.environ['KAGGLE_USERNAME'] = 'felipe1983'
os.environ['KAGGLE_KEY'] = '93dbcf741dba9085eded2cdbe2fc0cab'

SRC = Path('/content/kg1_adapter_v80')
DST = Path('/content/kg1_adapter_v80_stripped')

# IMPORTANTE: limpar diretório anterior pra garantir
if DST.exists():
    shutil.rmtree(DST)
DST.mkdir(parents=True, exist_ok=True)

# Também deleta zip antigo de 3GB
old_zip = Path('/content/kg1_out/submission_v80.zip')
if old_zip.exists():
    print(f'Deleting old zip: {old_zip.stat().st_size / 1024**2:.1f} MB')
    old_zip.unlink()

# === 1. Analyze safetensors keys ===
print('=== Analyzing adapter_model.safetensors (3GB) ===')
src_adapter = SRC / 'adapter_model.safetensors'
print(f'Source size: {src_adapter.stat().st_size / 1024**2:.1f} MB')

with safe_open(src_adapter, framework='pt') as f:
    all_keys = list(f.keys())

expert_keys = [k for k in all_keys if 'mlp.experts' in k]
base_keys = [k for k in all_keys if 'mlp.experts' not in k]
print(f'Total LoRA keys: {len(all_keys)}')
print(f'Expert LoRA keys (REMOVING): {len(expert_keys)}')
print(f'Base LoRA keys (KEEPING): {len(base_keys)}')
print(f'Sample kept keys:')
for k in base_keys[:3]:
    print(f'  {k}')
print(f'Sample removed keys:')
for k in expert_keys[:3]:
    print(f'  {k}')

# === 2. Strip MoE experts ===
print('\n=== Stripping MoE experts ===')
kept = {}
total_params_kept = 0
total_params_removed = 0

with safe_open(src_adapter, framework='pt') as f:
    for k in all_keys:
        t = f.get_tensor(k)
        numel = t.numel()
        if 'mlp.experts' in k:
            total_params_removed += numel
        else:
            kept[k] = t
            total_params_kept += numel

print(f'Kept params: {total_params_kept:,} ({total_params_kept/1e6:.1f}M)')
print(f'Removed params: {total_params_removed:,} ({total_params_removed/1e6:.1f}M)')
print(f'Saving stripped safetensors...')
save_file(kept, str(DST / 'adapter_model.safetensors'))

# === 3. Patch adapter_config.json ===
with open(SRC / 'adapter_config.json') as f:
    cfg = json.load(f)

# Force 8 targets (mesmo dgxchen v7)
cfg['target_modules'] = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                          'in_proj', 'out_proj', 'up_proj', 'down_proj']
cfg['base_model_name_or_path'] = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'
cfg['inference_mode'] = True
cfg['lora_dropout'] = 0.0
# Remove MoE-specific fields
for k in ['target_parameters', 'layers_to_transform', 'layers_pattern']:
    cfg.pop(k, None)

with open(DST / 'adapter_config.json', 'w') as f:
    json.dump(cfg, f, indent=2)

print(f'adapter_config.json: {cfg["target_modules"]}')

# === 4. Size check ===
orig_mb = src_adapter.stat().st_size / 1024**2
strip_mb = (DST / 'adapter_model.safetensors').stat().st_size / 1024**2
reduction = (1 - strip_mb/orig_mb) * 100
print(f'\nOriginal: {orig_mb:.1f} MB')
print(f'Stripped: {strip_mb:.1f} MB ({reduction:.1f}% reduction)')

# === 5. Build new zip ===
zip_path = Path('/content/kg1_out/submission_v80_stripped.zip')
zip_path.parent.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fn in ['adapter_config.json', 'adapter_model.safetensors']:
        zf.write(DST / fn, arcname=fn)

zip_mb = zip_path.stat().st_size / 1024**2
print(f'Zip stripped: {zip_mb:.1f} MB')

if zip_mb > 500:
    print(f'\n!!! STILL > 500 MB. Cannot submit.')
    print('Need deeper stripping or different approach.')
else:
    print(f'\nZip OK (< 500MB). Submitting...')

    # === 6. SUBMIT ===
    r = subprocess.run(
        ['kaggle', 'competitions', 'submit',
         '-c', 'nvidia-nemotron-model-reasoning-challenge',
         '-f', str(zip_path),
         '-m', f'V80 stripped MoE - 8 targets only - dgxchen v7 replica - min_loss=1.4275'],
        capture_output=True, text=True, timeout=600
    )
    print(f'\nSubmit rc={r.returncode}')
    print(r.stdout)
    if r.stderr:
        print(f'stderr: {r.stderr[-500:]}')

    # === 7. Verify ===
    print('\n=== Post-submit verification (wait 5s) ===')
    time.sleep(5)
    r = subprocess.run(['kaggle', 'competitions', 'submissions',
                        '-c', 'nvidia-nemotron-model-reasoning-challenge', '--csv'],
                       capture_output=True, text=True, timeout=60)
    print(r.stdout[:800])

print('\n=== DONE ===')
print('Check score: https://www.kaggle.com/competitions/nvidia-nemotron-model-reasoning-challenge/submissions')

Deleting old zip: 3093.1 MB
=== Analyzing adapter_model.safetensors (3GB) ===
Source size: 3373.4 MB
Total LoRA keys: 12008
Expert LoRA keys (REMOVING): 0
Base LoRA keys (KEEPING): 12008
Sample kept keys:
  base_model.model.backbone.layers.0.mixer.in_proj.lora_A.weight
  base_model.model.backbone.layers.0.mixer.in_proj.lora_B.weight
  base_model.model.backbone.layers.0.mixer.out_proj.lora_A.weight
Sample removed keys:

=== Stripping MoE experts ===
Kept params: 883,873,792 (883.9M)
Removed params: 0 (0.0M)
Saving stripped safetensors...
adapter_config.json: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'in_proj', 'out_proj', 'up_proj', 'down_proj']

Original: 3373.4 MB
Stripped: 3373.4 MB (0.0% reduction)
Zip stripped: 3093.1 MB

!!! STILL > 500 MB. Cannot submit.
Need deeper stripping or different approach.

=== DONE ===
Check score: https://www.kaggle.com/competitions/nvidia-nemotron-model-reasoning-challenge/submissions


In [13]:
# DIAGNÓSTICO: agrupar as 12008 keys por padrão intermediário
from safetensors import safe_open
from collections import Counter
import re

ADAPTER_PATH = "/content/kg1_out/sft_v80/checkpoint-245/adapter_model.safetensors"

with safe_open(ADAPTER_PATH, framework="pt") as f:
    all_keys = list(f.keys())

print(f"Total keys: {len(all_keys)}")
print()

# Extrair "middle path" entre backbone.layers.N. e .lora_A/B.weight
pattern_re = re.compile(r'base_model\.model\.(.+?)\.lora_[AB]\.weight$')
patterns = Counter()
for k in all_keys:
    m = pattern_re.search(k)
    if m:
        path = m.group(1)
        # Normaliza layer N -> layer_X para agrupar
        normalized = re.sub(r'layers\.\d+', 'layers.X', path)
        normalized = re.sub(r'experts\.\d+', 'experts.X', normalized)
        patterns[normalized] += 1

print("=== UNIQUE KEY PATTERNS (count = layers x 2 for lora_A+B) ===")
for pat, cnt in patterns.most_common():
    print(f"  {cnt:6d}  {pat}")

# Tamanho total por pattern
print()
print("=== SIZE PER PATTERN ===")
with safe_open(ADAPTER_PATH, framework="pt") as f:
    size_by_pattern = Counter()
    for k in all_keys:
        m = pattern_re.search(k)
        if m:
            path = m.group(1)
            normalized = re.sub(r'layers\.\d+', 'layers.X', path)
            normalized = re.sub(r'experts\.\d+', 'experts.X', normalized)
            tensor = f.get_tensor(k)
            size_by_pattern[normalized] += tensor.numel() * tensor.element_size()

for pat, size_b in sorted(size_by_pattern.items(), key=lambda x: -x[1]):
    size_mb = size_b / 1024**2
    print(f"  {size_mb:8.1f} MB  {pat}")

# Identificar automaticamente os padrões que contêm "experts" (os que devem ser strippados)
expert_patterns = [p for p in patterns if 'expert' in p.lower()]
print()
print(f"=== EXPERT PATTERNS DETECTED ({len(expert_patterns)}) ===")
for p in expert_patterns:
    print(f"  {p}  ->  {patterns[p]} keys, {size_by_pattern[p]/1024**2:.1f} MB")

# Proposta de filtro correto
print()
print("=== FILTRO PROPOSTO ===")
if expert_patterns:
    print("Use esta função para identificar expert keys:")
    print("  def is_expert(k): return any(marker in k for marker in ['experts.', 'mlp.expert'])")
else:
    print("Nenhum pattern com 'expert' encontrado — investigar manualmente os paths acima")

Total keys: 12008

=== UNIQUE KEY PATTERNS (count = layers x 2 for lora_A+B) ===
    5888  backbone.layers.X.mixer.experts.X.down_proj
    5888  backbone.layers.X.mixer.experts.X.up_proj
      46  backbone.layers.X.mixer.in_proj
      46  backbone.layers.X.mixer.out_proj
      46  backbone.layers.X.mixer.shared_experts.down_proj
      46  backbone.layers.X.mixer.shared_experts.up_proj
      12  backbone.layers.X.mixer.k_proj
      12  backbone.layers.X.mixer.o_proj
      12  backbone.layers.X.mixer.q_proj
      12  backbone.layers.X.mixer.v_proj

=== SIZE PER PATTERN ===
    1633.0 MB  backbone.layers.X.mixer.experts.X.down_proj
    1633.0 MB  backbone.layers.X.mixer.experts.X.up_proj
      36.5 MB  backbone.layers.X.mixer.in_proj
      19.0 MB  backbone.layers.X.mixer.out_proj
      18.0 MB  backbone.layers.X.mixer.shared_experts.down_proj
      18.0 MB  backbone.layers.X.mixer.shared_experts.up_proj
       5.0 MB  backbone.layers.X.mixer.o_proj
       5.0 MB  backbone.layers.X.mixer.

In [14]:
import os, json, subprocess
from google.colab import userdata

# Força reload do secret novo
os.environ['KAGGLE_USERNAME'] = 'felipe1983'
os.environ['KAGGLE_KEY'] = 'c0a9e0a7f303a43653b5ac47abed6028'

# Reescreve ~/.kaggle/kaggle.json com a nova chave
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump({'username': 'felipe1983', 'key': 'c0a9e0a7f303a43653b5ac47abed6028'}, f)
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

# Valida com chamada real
r = subprocess.run(
    ['kaggle', 'competitions', 'submissions',
     '-c', 'nvidia-nemotron-model-reasoning-challenge', '--csv'],
    capture_output=True, text=True, timeout=60,
)
if r.returncode == 0:
    print('[OK] Nova credencial validada')
    print(r.stdout[:800])
else:
    print('[FAIL] ' + r.stderr[:400])

[OK] Nova credencial validada
fileName,date,description,status,publicScore,privateScore
submission.zip,2026-04-20 12:34:51.980000,v18 stripped attempt3,SubmissionStatus.COMPLETE,0.50,
submission.zip,2026-04-20 12:33:08.957000,v18 stripped attempt2,SubmissionStatus.COMPLETE,0.50,
submission.zip,2026-04-20 12:31:26.853000,v18 stripped attempt1,SubmissionStatus.COMPLETE,0.50,
submission.zip,2026-04-20 12:30:08.120000,v18 stripped submission.zip,SubmissionStatus.COMPLETE,0.52,
submission.zip,2026-04-17 12:45:58.347000,v87 checkpoint-600 eval_loss 1.8460,SubmissionStatus.COMPLETE,0.54,
submission.zip,2026-04-16 12:10:55.310000,replica huikang v20 artifact hash C4EA449A,SubmissionStatus.COMPLETE,0.84,
submission.zip,2026-04-15 22:02:36.267000,v73 robusto A100-40GB Unsloth NF4 27M trainable loss 1.01 (avg last50 0.85),Submiss


In [15]:
import os, json, zipfile, hashlib, subprocess, glob

# 1. Localizar C4EA449A no GDrive
GDRIVE_SEARCH_ROOTS = [
    '/content/drive/MyDrive',
    '/content/drive/Shareddrives',
]

candidates = []
for root in GDRIVE_SEARCH_ROOTS:
    if not os.path.exists(root):
        continue
    for path in glob.glob(root + '/**/adapter_model.safetensors', recursive=True):
        try:
            with open(path, 'rb') as f:
                sha = hashlib.sha256(f.read()).hexdigest()
            if sha.upper().startswith('C4EA449A'):
                candidates.append((path, sha))
                print(f'[MATCH] {path}')
                print(f'        sha256={sha}')
        except Exception as e:
            continue

if not candidates:
    print('!!! C4EA449A NAO ENCONTRADO no GDrive')
    print('Buscar manualmente: find /content/drive -name adapter_model.safetensors 2>/dev/null')
else:
    # 2. Build zip + submit
    src_dir = os.path.dirname(candidates[0][0])
    sha_full = candidates[0][1]
    zip_path = '/content/v70_floor_resubmit.zip'

    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
        z.write(os.path.join(src_dir, 'adapter_config.json'), arcname='adapter_config.json')
        z.write(os.path.join(src_dir, 'adapter_model.safetensors'), arcname='adapter_model.safetensors')

    size_mb = os.path.getsize(zip_path) / 1024**2
    print(f'\n[ZIP] {zip_path} = {size_mb:.1f} MB')

    if size_mb > 500:
        print('!!! ZIP > 500MB, NAO submete')
    else:
        r = subprocess.run([
            'kaggle', 'competitions', 'submit',
            '-c', 'nvidia-nemotron-model-reasoning-challenge',
            '-f', zip_path,
            '-m', f'V70 floor healthcheck sha={sha_full[:12]}',
        ], capture_output=True, text=True, timeout=300)

        print(f'\nReturncode: {r.returncode}')
        print(f'stdout: {r.stdout[:500]}')
        print(f'stderr: {r.stderr[:500]}')

!!! C4EA449A NAO ENCONTRADO no GDrive
Buscar manualmente: find /content/drive -name adapter_model.safetensors 2>/dev/null


In [16]:
import os, hashlib, json, glob, subprocess

SEARCH_ROOTS = [
    '/content/drive/MyDrive',
    '/content/drive/Shareddrives',
    '/content',  # incluindo /content/kg1_out, /content/kg1_adapter_v80, etc
]

candidates = []

print('=== Buscando adapter_model.safetensors > 30MB ===\n')

for root in SEARCH_ROOTS:
    if not os.path.exists(root):
        continue
    for path in glob.glob(root + '/**/adapter_model.safetensors', recursive=True):
        try:
            size_mb = os.path.getsize(path) / 1024**2
            if size_mb < 30:
                continue

            folder = os.path.dirname(path)
            cfg_path = os.path.join(folder, 'adapter_config.json')
            cfg_info = {}
            if os.path.exists(cfg_path):
                with open(cfg_path) as f:
                    cfg = json.load(f)
                cfg_info = {
                    'r': cfg.get('r'),
                    'alpha': cfg.get('lora_alpha'),
                    'targets': cfg.get('target_modules', [])[:5],
                    'n_targets': len(cfg.get('target_modules', [])) if isinstance(cfg.get('target_modules'), list) else 'regex',
                }

            # SHA256 do adapter (rápido com chunk streaming)
            h = hashlib.sha256()
            with open(path, 'rb') as f:
                while chunk := f.read(1024*1024):
                    h.update(chunk)
            sha = h.hexdigest()

            is_c4ea = sha.upper().startswith('C4EA449A')

            candidates.append({
                'path': path,
                'folder': folder,
                'size_mb': round(size_mb, 1),
                'sha256': sha,
                'sha_prefix': sha[:12].upper(),
                'is_c4ea449a': is_c4ea,
                'cfg': cfg_info,
            })

            flag = ' !!! V70 C4EA449A EXACT MATCH !!!' if is_c4ea else ''
            print(f'{"="*70}')
            print(f'Path:    {path}')
            print(f'Size:    {size_mb:.1f} MB')
            print(f'SHA256:  {sha[:16]}...{flag}')
            if cfg_info:
                print(f'Config:  r={cfg_info["r"]} alpha={cfg_info["alpha"]} targets={cfg_info["n_targets"]} modules')
                print(f'         first_5={cfg_info["targets"]}')
            print()
        except Exception as e:
            print(f'[ERR] {path}: {str(e)[:80]}')

print(f'\n=== RESUMO ===')
print(f'Total candidatos: {len(candidates)}')

# Triage
print(f'\nCandidatos ordenados por probabilidade de ser V70 floor 0.84:')
def v70_score(c):
    score = 0
    if c['is_c4ea449a']: score += 1000
    cfg = c.get('cfg', {})
    if cfg.get('r') == 32 and cfg.get('alpha') == 32: score += 100
    if 100 <= c['size_mb'] <= 200: score += 50  # V70 stripped típico
    if 'v70' in c['path'].lower() or 'huikang' in c['path'].lower(): score += 80
    if 'final' in c['path'].lower(): score += 20
    return -score

for i, c in enumerate(sorted(candidates, key=v70_score)[:10]):
    print(f'{i+1}. {c["path"]}')
    print(f'   {c["size_mb"]:.1f}MB sha={c["sha_prefix"]} r={c["cfg"].get("r")} α={c["cfg"].get("alpha")}')
    if c['is_c4ea449a']: print(f'   *** V70 EXACT MATCH ***')

# Também lista HF repos do usuário (adapter pode ter feito upload)
print(f'\n=== HF repos do felipesp1983 ===')
try:
    from huggingface_hub import HfApi
    api = HfApi(token=os.environ.get('HF_TOKEN'))
    for repo in api.list_models(author='felipesp1983', limit=50):
        name = repo.modelId
        if any(k in name.lower() for k in ['v70', 'huikang', 'nemotron', 'lora']):
            print(f'  hf://{name}  (modified: {repo.lastModified})')
except Exception as e:
    print(f'  [ERR] {str(e)[:100]}')

=== Buscando adapter_model.safetensors > 30MB ===

Path:    /content/kg1_out/sft_v80/checkpoint-245/adapter_model.safetensors
Size:    3373.4 MB
SHA256:  a04d901656330d9c...
Config:  r=32 alpha=32 targets=8 modules
         first_5=['out_proj', 'o_proj', 'v_proj', 'in_proj', 'up_proj']

Path:    /content/kg1_out/sft_v80/checkpoint-200/adapter_model.safetensors
Size:    3373.4 MB
SHA256:  556b0889a0b8ada8...
Config:  r=32 alpha=32 targets=8 modules
         first_5=['out_proj', 'o_proj', 'v_proj', 'in_proj', 'up_proj']

Path:    /content/kg1_out/sft_v80/checkpoint-150/adapter_model.safetensors
Size:    3373.4 MB
SHA256:  2159424c6be8dffe...
Config:  r=32 alpha=32 targets=8 modules
         first_5=['out_proj', 'o_proj', 'v_proj', 'in_proj', 'up_proj']

Path:    /content/kg1_out/submission_v80/adapter_model.safetensors
Size:    3373.4 MB
SHA256:  a04d901656330d9c...
Config:  r=32 alpha=32 targets=8 modules
         first_5=['out_proj', 'o_proj', 'v_proj', 'in_proj', 'up_proj']

Path:    

In [17]:
import os, hashlib, json, glob, subprocess

SEARCH_ROOTS = [
    '/content/drive/MyDrive',
    '/content/drive/Shareddrives',
    '/content',  # incluindo /content/kg1_out, /content/kg1_adapter_v80, etc
]

candidates = []

print('=== Buscando adapter_model.safetensors > 30MB ===\n')

for root in SEARCH_ROOTS:
    if not os.path.exists(root):
        continue
    for path in glob.glob(root + '/**/adapter_model.safetensors', recursive=True):
        try:
            size_mb = os.path.getsize(path) / 1024**2
            if size_mb < 30:
                continue

            folder = os.path.dirname(path)
            cfg_path = os.path.join(folder, 'adapter_config.json')
            cfg_info = {}
            if os.path.exists(cfg_path):
                with open(cfg_path) as f:
                    cfg = json.load(f)
                cfg_info = {
                    'r': cfg.get('r'),
                    'alpha': cfg.get('lora_alpha'),
                    'targets': cfg.get('target_modules', [])[:5],
                    'n_targets': len(cfg.get('target_modules', [])) if isinstance(cfg.get('target_modules'), list) else 'regex',
                }

            # SHA256 do adapter (rápido com chunk streaming)
            h = hashlib.sha256()
            with open(path, 'rb') as f:
                while chunk := f.read(1024*1024):
                    h.update(chunk)
            sha = h.hexdigest()

            is_c4ea = sha.upper().startswith('C4EA449A')

            candidates.append({
                'path': path,
                'folder': folder,
                'size_mb': round(size_mb, 1),
                'sha256': sha,
                'sha_prefix': sha[:12].upper(),
                'is_c4ea449a': is_c4ea,
                'cfg': cfg_info,
            })

            flag = ' !!! V70 C4EA449A EXACT MATCH !!!' if is_c4ea else ''
            print(f'{"="*70}')
            print(f'Path:    {path}')
            print(f'Size:    {size_mb:.1f} MB')
            print(f'SHA256:  {sha[:16]}...{flag}')
            if cfg_info:
                print(f'Config:  r={cfg_info["r"]} alpha={cfg_info["alpha"]} targets={cfg_info["n_targets"]} modules')
                print(f'         first_5={cfg_info["targets"]}')
            print()
        except Exception as e:
            print(f'[ERR] {path}: {str(e)[:80]}')

print(f'\n=== RESUMO ===')
print(f'Total candidatos: {len(candidates)}')

# Triage
print(f'\nCandidatos ordenados por probabilidade de ser V70 floor 0.84:')
def v70_score(c):
    score = 0
    if c['is_c4ea449a']: score += 1000
    cfg = c.get('cfg', {})
    if cfg.get('r') == 32 and cfg.get('alpha') == 32: score += 100
    if 100 <= c['size_mb'] <= 200: score += 50  # V70 stripped típico
    if 'v70' in c['path'].lower() or 'huikang' in c['path'].lower(): score += 80
    if 'final' in c['path'].lower(): score += 20
    return -score

for i, c in enumerate(sorted(candidates, key=v70_score)[:10]):
    print(f'{i+1}. {c["path"]}')
    print(f'   {c["size_mb"]:.1f}MB sha={c["sha_prefix"]} r={c["cfg"].get("r")} α={c["cfg"].get("alpha")}')
    if c['is_c4ea449a']: print(f'   *** V70 EXACT MATCH ***')

# Também lista HF repos do usuário (adapter pode ter feito upload)
print(f'\n=== HF repos do felipesp1983 ===')
try:
    from huggingface_hub import HfApi
    api = HfApi(token=os.environ.get('HF_TOKEN'))
    for repo in api.list_models(author='felipesp1983', limit=50):
        name = repo.modelId
        if any(k in name.lower() for k in ['v70', 'huikang', 'nemotron', 'lora']):
            print(f'  hf://{name}  (modified: {repo.lastModified})')
except Exception as e:
    print(f'  [ERR] {str(e)[:100]}')

=== Buscando adapter_model.safetensors > 30MB ===

Path:    /content/kg1_out/sft_v80/checkpoint-245/adapter_model.safetensors
Size:    3373.4 MB
SHA256:  a04d901656330d9c...
Config:  r=32 alpha=32 targets=8 modules
         first_5=['out_proj', 'o_proj', 'v_proj', 'in_proj', 'up_proj']

Path:    /content/kg1_out/sft_v80/checkpoint-200/adapter_model.safetensors
Size:    3373.4 MB
SHA256:  556b0889a0b8ada8...
Config:  r=32 alpha=32 targets=8 modules
         first_5=['out_proj', 'o_proj', 'v_proj', 'in_proj', 'up_proj']

Path:    /content/kg1_out/sft_v80/checkpoint-150/adapter_model.safetensors
Size:    3373.4 MB
SHA256:  2159424c6be8dffe...
Config:  r=32 alpha=32 targets=8 modules
         first_5=['out_proj', 'o_proj', 'v_proj', 'in_proj', 'up_proj']

Path:    /content/kg1_out/submission_v80/adapter_model.safetensors
Size:    3373.4 MB
SHA256:  a04d901656330d9c...
Config:  r=32 alpha=32 targets=8 modules
         first_5=['out_proj', 'o_proj', 'v_proj', 'in_proj', 'up_proj']

Path:    

In [18]:
import os, json, shutil, zipfile, hashlib, subprocess, re
from safetensors import safe_open
from safetensors.torch import save_file

SRC = '/content/kg1_out/sft_v80/checkpoint-245'
DST = '/content/kg1_v80_stripped_v2'

shutil.rmtree(DST, ignore_errors=True)
os.makedirs(DST, exist_ok=True)

# 1. Strip experts CORRETAMENTE
print('=== Stripping experts.N.up_proj + experts.N.down_proj ===')
src_file = os.path.join(SRC, 'adapter_model.safetensors')

expert_re = re.compile(r'\.experts\.\d+\.(up_proj|down_proj)\.')

kept_tensors = {}
total_keys = 0
kept_keys = 0
stripped_keys = 0
stripped_bytes = 0
kept_bytes = 0

with safe_open(src_file, framework='pt') as f:
    for k in f.keys():
        total_keys += 1
        t = f.get_tensor(k)
        nbytes = t.numel() * t.element_size()
        if expert_re.search(k):
            stripped_keys += 1
            stripped_bytes += nbytes
        else:
            kept_keys += 1
            kept_bytes += nbytes
            kept_tensors[k] = t

print(f'  Total keys:     {total_keys}')
print(f'  Stripped keys:  {stripped_keys} ({stripped_bytes/1024**2:.1f} MB)')
print(f'  Kept keys:      {kept_keys} ({kept_bytes/1024**2:.1f} MB)')

# Sanity: show sample kept
print('\nSample kept keys (first 5):')
for k in list(kept_tensors.keys())[:5]:
    print(f'  {k}')

# 2. Salvar safetensors strippado
dst_file = os.path.join(DST, 'adapter_model.safetensors')
save_file(kept_tensors, dst_file)
dst_size = os.path.getsize(dst_file) / 1024**2
print(f'\nSaved: {dst_file} = {dst_size:.1f} MB')

# 3. Patch adapter_config.json
src_cfg = json.load(open(os.path.join(SRC, 'adapter_config.json')))

# Identificar target_modules do V80 e manter (são os 8 modules base)
print(f'\nV80 target_modules: {src_cfg["target_modules"]}')

# V80 adapter_config NÃO muda - o peft vai identificar só os modules presentes no safetensors
# mas IMPORTANTE: precisamos GARANTIR que inference não tente criar LoRA nos 128 experts
# Isso é controlado por target_modules:
# - Se target_modules='all-linear' OU contém up_proj/down_proj suffix, PEFT vai criar LoRA
#   em TODOS os up_proj/down_proj incluindo experts → falha (weights missing)
# - Solução: usar regex target_modules que EXCLUI experts

new_cfg = dict(src_cfg)

# Forçar target_modules como REGEX precisa (exclude .experts.N.)
# Matches: q/k/v/o_proj, in_proj, out_proj, shared_experts.up/down_proj, lm_head
new_cfg['target_modules'] = r".*\.(q_proj|k_proj|v_proj|o_proj|in_proj|out_proj)|.*\.shared_experts\.(up_proj|down_proj)|lm_head"

# Sanity: base_model_name_or_path precisa estar set para Kaggle
if not new_cfg.get('base_model_name_or_path'):
    new_cfg['base_model_name_or_path'] = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'
new_cfg['inference_mode'] = True

dst_cfg = os.path.join(DST, 'adapter_config.json')
with open(dst_cfg, 'w') as f:
    json.dump(new_cfg, f, indent=2)
print(f'Patched config: {dst_cfg}')
print(f'  target_modules = {new_cfg["target_modules"]}')

# 4. Build zip
zip_path = '/content/v80_stripped_v2.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(dst_cfg, arcname='adapter_config.json')
    z.write(dst_file, arcname='adapter_model.safetensors')

zip_mb = os.path.getsize(zip_path) / 1024**2
with open(zip_path, 'rb') as f:
    zsha = hashlib.sha256(f.read()).hexdigest()
print(f'\nZIP: {zip_path} = {zip_mb:.1f} MB (sha={zsha[:12]})')

if zip_mb > 500:
    print(f'!!! ZIP > 500MB — NAO submete')
else:
    print(f'[OK] ZIP fits 500MB limit')

    # 5. Submit Kaggle
    print('\n=== Submetendo ao Kaggle ===')
    r = subprocess.run([
        'kaggle', 'competitions', 'submit',
        '-c', 'nvidia-nemotron-model-reasoning-challenge',
        '-f', zip_path,
        '-m', f'V80 stripped v2 (correct experts pattern) sha={zsha[:12]}',
    ], capture_output=True, text=True, timeout=300)

    print(f'rc={r.returncode}')
    print(f'stdout: {r.stdout[:500]}')
    print(f'stderr: {r.stderr[:500]}')

    if r.returncode == 0:
        print('\n[OK] V80 stripped v2 submetido. Aguarda ~10 min pro score.')
        print('Check: kaggle competitions submissions -c nvidia-nemotron-model-reasoning-challenge | head -3')

=== Stripping experts.N.up_proj + experts.N.down_proj ===
  Total keys:     12008
  Stripped keys:  11776 (3266.0 MB)
  Kept keys:      232 (105.7 MB)

Sample kept keys (first 5):
  base_model.model.backbone.layers.0.mixer.in_proj.lora_A.weight
  base_model.model.backbone.layers.0.mixer.in_proj.lora_B.weight
  base_model.model.backbone.layers.0.mixer.out_proj.lora_A.weight
  base_model.model.backbone.layers.0.mixer.out_proj.lora_B.weight
  base_model.model.backbone.layers.1.mixer.shared_experts.down_proj.lora_A.weight

Saved: /content/kg1_v80_stripped_v2/adapter_model.safetensors = 105.7 MB

V80 target_modules: ['out_proj', 'o_proj', 'v_proj', 'in_proj', 'up_proj', 'q_proj', 'k_proj', 'down_proj']
Patched config: /content/kg1_v80_stripped_v2/adapter_config.json
  target_modules = .*\.(q_proj|k_proj|v_proj|o_proj|in_proj|out_proj)|.*\.shared_experts\.(up_proj|down_proj)|lm_head

ZIP: /content/v80_stripped_v2.zip = 85.2 MB (sha=ac54a81bb3e8)
[OK] ZIP fits 500MB limit

=== Submetendo ao K

In [19]:
import os, json, zipfile, hashlib, subprocess, shutil

DST = '/content/kg1_v80_stripped_v2'

# 1. Fix adapter_config.json: usar lista igual V70 (SEM gate_proj forbidden)
cfg_path = os.path.join(DST, 'adapter_config.json')
cfg = json.load(open(cfg_path))

# V70 exact target_modules (que scora 0.84):
# Ordem do V70: ["k_proj","o_proj","in_proj","q_proj","up_proj","v_proj","down_proj","out_proj","lm_head"]
cfg['target_modules'] = [
    'k_proj', 'o_proj', 'in_proj', 'q_proj',
    'up_proj', 'v_proj', 'down_proj', 'out_proj',
    'lm_head',
]
cfg['base_model_name_or_path'] = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'
cfg['inference_mode'] = True
cfg['lora_dropout'] = 0.0
cfg['r'] = 32
cfg['lora_alpha'] = 32

with open(cfg_path, 'w') as f:
    json.dump(cfg, f, indent=2)
print(f'[OK] config patched: {cfg["target_modules"]}')

# 2. Rebuild zip
zip_path = '/content/v80_stripped_v3.zip'
if os.path.exists(zip_path):
    os.remove(zip_path)
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(cfg_path, arcname='adapter_config.json')
    z.write(os.path.join(DST, 'adapter_model.safetensors'), arcname='adapter_model.safetensors')

zip_mb = os.path.getsize(zip_path) / 1024**2
with open(zip_path, 'rb') as f:
    zsha = hashlib.sha256(f.read()).hexdigest()
print(f'ZIP v3: {zip_path} = {zip_mb:.1f} MB (sha={zsha[:12]})')

# 3. Verificar submissions pendentes antes de submit
print('\n=== Submissions recentes (ultimas 5) ===')
r_check = subprocess.run([
    'kaggle', 'competitions', 'submissions',
    '-c', 'nvidia-nemotron-model-reasoning-challenge', '--csv'
], capture_output=True, text=True, timeout=60)
for line in r_check.stdout.split('\n')[:6]:
    print(f'  {line}')

print('\n=== Count de submissions hoje ===')
import csv, io
from datetime import datetime, timezone
today = datetime.now(timezone.utc).date()
cnt = 0
for row in csv.DictReader(io.StringIO(r_check.stdout)):
    try:
        d = datetime.strptime(row['date'][:10], '%Y-%m-%d').date()
        if d == today:
            cnt += 1
    except Exception:
        continue
print(f'  Submissions hoje (UTC): {cnt}/5')

if cnt >= 5:
    print('!!! Daily limit atingido. Aguarda 00:00 UTC reset.')
else:
    # 4. Retry submit via CLI direto (mais robusto que subprocess)
    print('\n=== Retry submit ===')
    import subprocess
    result = subprocess.run(
        ['kaggle', 'competitions', 'submit',
         '-c', 'nvidia-nemotron-model-reasoning-challenge',
         '-f', zip_path,
         '-m', f'V80 stripped v3 list_targets sha={zsha[:12]}'],
        capture_output=True, text=True, timeout=600,
    )
    print(f'rc={result.returncode}')
    print(f'stdout: {result.stdout[:800]}')
    print(f'stderr: {result.stderr[:800]}')

[OK] config patched: ['k_proj', 'o_proj', 'in_proj', 'q_proj', 'up_proj', 'v_proj', 'down_proj', 'out_proj', 'lm_head']
ZIP v3: /content/v80_stripped_v3.zip = 85.2 MB (sha=e45cabfc5d71)

=== Submissions recentes (ultimas 5) ===
  fileName,date,description,status,publicScore,privateScore
  submission.zip,2026-04-20 12:34:51.980000,v18 stripped attempt3,SubmissionStatus.COMPLETE,0.50,
  submission.zip,2026-04-20 12:33:08.957000,v18 stripped attempt2,SubmissionStatus.COMPLETE,0.50,
  submission.zip,2026-04-20 12:31:26.853000,v18 stripped attempt1,SubmissionStatus.COMPLETE,0.50,
  submission.zip,2026-04-20 12:30:08.120000,v18 stripped submission.zip,SubmissionStatus.COMPLETE,0.52,
  submission.zip,2026-04-17 12:45:58.347000,v87 checkpoint-600 eval_loss 1.8460,SubmissionStatus.COMPLETE,0.54,

=== Count de submissions hoje ===
  Submissions hoje (UTC): 0/5

=== Retry submit ===
rc=1
stdout: 400 Client Error: Bad Request for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/Cr

In [20]:
import os, subprocess, json, zipfile, hashlib

# =========================================================
# DIAGNOSTIC 1: Auth OK? (comando LIST deve funcionar)
# =========================================================
print('=== DIAG 1: Auth check via competitions list ===')
r = subprocess.run(['kaggle', 'competitions', 'list', '-m'],
                    capture_output=True, text=True, timeout=30)
print(f'rc={r.returncode}')
print(r.stdout[:300] if r.returncode == 0 else r.stderr[:300])

# =========================================================
# DIAGNOSTIC 2: Kaggle CLI version
# =========================================================
print('\n=== DIAG 2: Kaggle version ===')
r = subprocess.run(['kaggle', '--version'], capture_output=True, text=True)
print(r.stdout.strip())

# =========================================================
# DIAGNOSTIC 3: Inspect current zip content
# =========================================================
print('\n=== DIAG 3: ZIP v3 content ===')
with zipfile.ZipFile('/content/v80_stripped_v3.zip', 'r') as z:
    for info in z.infolist():
        print(f'  {info.filename}: size={info.file_size} compressed={info.compress_size}')

# =========================================================
# FIX 1: Upgrade kaggle CLI (maybe old version has bug)
# =========================================================
print('\n=== FIX 1: Upgrade kaggle CLI ===')
!pip install --upgrade --quiet kaggle 2>&1 | tail -3

# =========================================================
# FIX 2: Rebuild as submission.zip STORED (no compression)
# =========================================================
print('\n=== FIX 2: Rebuild submission.zip ZIP_STORED (no compression) ===')
DST = '/content/kg1_v80_stripped_v2'
new_zip = '/content/submission.zip'  # Kaggle prefere esse nome
if os.path.exists(new_zip):
    os.remove(new_zip)

with zipfile.ZipFile(new_zip, 'w', zipfile.ZIP_STORED) as z:
    z.write(os.path.join(DST, 'adapter_config.json'), arcname='adapter_config.json')
    z.write(os.path.join(DST, 'adapter_model.safetensors'), arcname='adapter_model.safetensors')

size_mb = os.path.getsize(new_zip) / 1024**2
with open(new_zip, 'rb') as f:
    sha = hashlib.sha256(f.read()).hexdigest()
print(f'  {new_zip} = {size_mb:.1f} MB (sha={sha[:12]})')

if size_mb > 500:
    print(f'!!! STORED > 500MB, uso DEFLATE como fallback')
    with zipfile.ZipFile(new_zip, 'w', zipfile.ZIP_DEFLATED) as z:
        z.write(os.path.join(DST, 'adapter_config.json'), arcname='adapter_config.json')
        z.write(os.path.join(DST, 'adapter_model.safetensors'), arcname='adapter_model.safetensors')
    size_mb = os.path.getsize(new_zip) / 1024**2
    print(f'  Fallback DEFLATE = {size_mb:.1f} MB')

# =========================================================
# FIX 3: Submit via Kaggle Python API (bypass CLI)
# =========================================================
print('\n=== FIX 3: Submit via Python API ===')
try:
    # Force fresh auth
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()

    result = api.competition_submit(
        file_name=new_zip,
        message=f'V80 stripped submission.zip stored sha={sha[:12]}',
        competition='nvidia-nemotron-model-reasoning-challenge',
    )
    print('[OK] Python API submit:')
    print(result)
except Exception as e:
    print(f'[FAIL Python API] {type(e).__name__}: {str(e)[:500]}')

    # =========================================================
    # FALLBACK: CLI with upgraded kaggle + new filename
    # =========================================================
    print('\n=== FALLBACK: CLI with new submission.zip ===')
    r = subprocess.run([
        'kaggle', 'competitions', 'submit',
        '-c', 'nvidia-nemotron-model-reasoning-challenge',
        '-f', new_zip,
        '-m', f'V80 stripped final sha={sha[:12]}',
    ], capture_output=True, text=True, timeout=900)
    print(f'rc={r.returncode}')
    print(f'stdout: {r.stdout[:600]}')
    print(f'stderr: {r.stderr[-600:]}')

=== DIAG 1: Auth check via competitions list ===
rc=2
usage: kaggle [-h] [-v] [-W]
              {competitions,c,datasets,d,kernels,k,models,m,files,f,config,auth}
              ...
kaggle: error: unrecognized arguments: -m


=== DIAG 2: Kaggle version ===
Kaggle CLI 2.0.1

=== DIAG 3: ZIP v3 content ===
  adapter_config.json: size=1187 compressed=513
  adapter_model.safetensors: size=110878712 compressed=89371742

=== FIX 1: Upgrade kaggle CLI ===

=== FIX 2: Rebuild submission.zip ZIP_STORED (no compression) ===
  /content/submission.zip = 105.7 MB (sha=ab100039cb48)

=== FIX 3: Submit via Python API ===


100%|██████████| 106M/106M [00:10<00:00, 11.0MB/s]


[OK] Python API submit:
{"message": "Successfully submitted to NVIDIA Nemotron Model Reasoning Challenge", "ref": 51966159}


In [21]:
import subprocess, csv, io
r = subprocess.run(
    ['kaggle', 'competitions', 'submissions',
     '-c', 'nvidia-nemotron-model-reasoning-challenge', '--csv'],
    capture_output=True, text=True, timeout=60
)
# Pega só as últimas 3 submissions
for i, row in enumerate(csv.DictReader(io.StringIO(r.stdout))):
    if i >= 3: break
    status = row.get('status', '')
    score = row.get('publicScore', '') or 'pending'
    date = row.get('date', '')[:16]
    desc = row.get('description', '')[:60]
    print(f'{date} | {status} | score={score} | {desc}')

2026-04-23 11:58 | SubmissionStatus.PENDING | score=pending | V80 stripped submission.zip stored sha=ab100039cb48
2026-04-20 12:34 | SubmissionStatus.COMPLETE | score=0.50 | v18 stripped attempt3
2026-04-20 12:33 | SubmissionStatus.COMPLETE | score=0.50 | v18 stripped attempt2


In [22]:
import subprocess, csv, io, time

TARGET_SHA = 'ab100039cb48'  # V80 stripped que submetemos

def check_once():
    r = subprocess.run(
        ['kaggle', 'competitions', 'submissions',
         '-c', 'nvidia-nemotron-model-reasoning-challenge', '--csv'],
        capture_output=True, text=True, timeout=60
    )
    for row in csv.DictReader(io.StringIO(r.stdout)):
        desc = row.get('description', '')
        if TARGET_SHA in desc:
            return {
                'status': row.get('status', ''),
                'score': row.get('publicScore', '') or 'pending',
                'date': row.get('date', '')[:19],
                'desc': desc[:80],
            }
    return None

# Polling a cada 60s, max 30 min
print(f'=== Polling V80 stripped (sha={TARGET_SHA}) ===')
for attempt in range(30):
    res = check_once()
    if res is None:
        print(f'[{attempt+1}] Submission not found in list (weird)')
    elif res['status'] == 'SubmissionStatus.COMPLETE':
        print(f'\n*** SCORE READY ***')
        print(f'  Status:  {res["status"]}')
        print(f'  Score:   {res["score"]}')
        print(f'  Date:    {res["date"]}')
        print(f'  Desc:    {res["desc"]}')
        break
    elif 'FAIL' in res['status'] or 'ERROR' in res['status']:
        print(f'\n!!! FAILED ***')
        print(f'  Status: {res["status"]}')
        print(f'  Desc:   {res["desc"]}')
        break
    else:
        print(f'[{attempt+1}] status={res["status"]} score={res["score"]}')
    time.sleep(60)
else:
    print('Timeout 30 min — check manualmente depois')

=== Polling V80 stripped (sha=ab100039cb48) ===
[1] status=SubmissionStatus.PENDING score=pending
[2] status=SubmissionStatus.PENDING score=pending
[3] status=SubmissionStatus.PENDING score=pending
[4] status=SubmissionStatus.PENDING score=pending
[5] status=SubmissionStatus.PENDING score=pending
[6] status=SubmissionStatus.PENDING score=pending
[7] status=SubmissionStatus.PENDING score=pending
[8] status=SubmissionStatus.PENDING score=pending
[9] status=SubmissionStatus.PENDING score=pending
[10] status=SubmissionStatus.PENDING score=pending
[11] status=SubmissionStatus.PENDING score=pending
[12] status=SubmissionStatus.PENDING score=pending
[13] status=SubmissionStatus.PENDING score=pending
[14] status=SubmissionStatus.PENDING score=pending
[15] status=SubmissionStatus.PENDING score=pending
[16] status=SubmissionStatus.PENDING score=pending
[17] status=SubmissionStatus.PENDING score=pending
[18] status=SubmissionStatus.PENDING score=pending
[19] status=SubmissionStatus.PENDING score=